# பாடம் 01 - AI முகவரிகளுக்கான அறிமுகம்

**ஆரம்பக்காரர்களுக்கான AI முகவர்கள்** பாடத்தின் முதல் பாடத்துக்கு வருக!

**AI முகவர்** என்பது பெரிய மொழி மாதிரியை (LLM) அதனுடைய காரணம்செய்தி இயந்திரமாக பயன்படுத்தி, *நடவடிக்கைகள்* மேற்கொள்ளக்கூடிய நிகழ்ச்சி ஆகும் — APIகளை அழைத்தல், தரவுத்தளங்களை கேட்குதல், அல்லது குறியீட்டை இயக்குதல் — பயனரின் சார்பாக ஒரு இலக்கை அடைவதற்கு.

இந்த நோட்புக்கில் நீங்கள் உங்கள் முதல் முகவரான ஒரு **பயண முகவர்** உருவாக்குவீர்கள், இது விடுமுறை இடங்களை பரிந்துரைக்கும். இதைச் செய்யும் போது நீங்கள் கற்றுக்கொள்ளப்போகும் விஷயங்கள்:

1. **Microsoft Agent Framework** பயன்படுத்தி Azure AI Foundry Agent சேவையுடன் இணைப்பது.
2. முகவருக்கு **கருவி** கொடுப்பது — அது அழைக்கக்கூடிய சாதாரண Python செயல்பாடு.
3. முகவரைக் இயக்கி அதன் பதிலை பரிசீலித்தல்.
4. முகவரின் பதிலை அகராதி ஒன்றுக்கு அகராதி வாரியாக ஸ்ட்ரீம் செய்வது.


## அமைப்பு

இந்த நோட்புக் ஐ இயக்குவதற்கு முன், உங்களுக்கு மீண்டும் உறுதி செய்யவும்:

1. **வெற்றிகரமாக நிறுவப்பட்ட உரையாடல் மாதிரியுடன் ஒரு Azure AI Foundry திட்டம்** (எ.கா., `gpt-4o-mini`).
2. **Azure CLI உடன் உள்நுழைந்திருக்கவும்** — உங்கள் டெர்மினலில் `az login` ஐ இயக்கவும்.
3. **தேவையான சுற்றுச்சூழல் மாறிகளை அமைத்திருக்கவும்:**
   - `AZURE_AI_PROJECT_ENDPOINT` — உங்கள் Azure AI Foundry திட்ட அனுப்புநிலை URL.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — உங்கள் நிறுவப்பட்ட மாதிரி பெயர்.

கீழே உள்ள செல்அடை Python தொகுப்புகளை நிறுவுகிறது.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## உங்கள் முதல் முகவரியை உருவாக்குதல்

ஒரு முகவரிக்கு இரண்டு விஷயங்கள் தேவை:

- அதற்கு *யார் என்று* மற்றும் *எப்படி நடக்க வேண்டும்* என்பதைச் சொல்லும் **வழிமுறைகள்** (ஒரு அமைப்பு அறிக்கை).
- முகவர் தகவல் பெற அல்லது செயல்கள் செய்ய அழைக்கக் கூடிய `@tool` என்ற அலங்கரிப்புடன் கூடிய Python செயல்பாடுகள் — **கருவிகள்**.

கீழே பயண பரிந்துரைகள் கேட்டபோது முகவர் பயன்படுத்தும் பபுளர் விடுமுறை இடங்களின் பட்டியலை தரும் ஒரு எளிய கருவி வரையறுக்கப்பட்டுள்ளது.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## ஸ்ட்ரீமிங் பதில்கள்

மேலும் இன்டராக்டிவான அனுபவத்தை பெற நீங்கள் முகவரின் பதிலை **ஸ்ட்ரீம்** செய்யலாம். முழு பதிலை காத்திருக்காமல், முகவரி உருவாக்கும் போது உரை பகுதிகளை தருகிறது. இது நேரடி வெளிப்பாட்டை காட்ட விரும்பும் உரையாடல் இடைமுகங்களில் மிகவும் பயனுள்ளது.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## சுருக்கம்

இந்த பாடத்தில் நீங்கள் எப்படி செய்ய வேண்டும் என்பதை கற்றுக்கொண்டீர்கள்:

- **ஒரு சேவையகத்தை உருவாக்கு** `AzureAIProjectAgentProvider` மூலம் Azure AI Foundry Agent Service-க்கு இணைக்க.
- **ஒரு கருவியை வரையறு** `@tool` அலங்காரியை பயன்படுத்தி, ஆகென்ட் உங்கள் Python செயல்பாடுகளை அழைக்கக் கூடியதாக.
- **ஆகெண்ட் இயக்கவும்** ஒரு பயனர் செய்தியுடன் அதன் பதிலை அச்சிடவும்.
- **உடனடி வெளியீட்டிற்கு பதில்கள் ஸ்ட்ரீம் செய்யவும்**.

அடுத்த பாடத்தில் நாம் ஆகெண்ட் அடிப்படைக் கட்டமைப்புகளை விரிவாக ஆராய்ந்து, ஆகெண்ட்களுக்கு அதிக சக்திவாய்ந்த கருவிகள் மற்றும் பல படி காரணிப்பாட்டு திறன்களை வழங்க கற்றுக்கொள்வோம்.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
